# Instance-Aware Depth on Colab

Runs the project's code on a Colab GPU. **Runtime → Change runtime type → T4 GPU** first.

**Data** (first match wins):
1. `MyDrive/instance-aware-depth/iad_data.tar` (your own copy), or
2. `MyDrive/iad_data.tar`: a **shortcut to the team's shared, view-only bundle**. In Drive, open
   "Shared with me", right-click `iad_data.tar` → Organize → Add shortcut → My Drive.
3. Neither: the notebook downloads and prepares everything (~10 min) and saves your own copy.

Checkpoints and results are always saved to **your own** `MyDrive/instance-aware-depth/`, never to the shared file.

**Private repo:** add a GitHub token as a Colab secret named `GH_TOKEN` (🔑 icon on the left).
Not needed once the repo is public.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; print('torch', torch.__version__, '| CUDA', torch.cuda.is_available())

## 1. Settings and Google Drive

In [ ]:
import os
REPO = 'csgomez25/instance-aware-depth'
BRANCH = 'main'
DRIVE_DIR = '/content/drive/MyDrive/instance-aware-depth'   # YOUR folder: checkpoints, results, own data copy

from google.colab import drive
drive.mount('/content/drive')
os.makedirs(DRIVE_DIR, exist_ok=True)
os.environ['DATA_ROOT'] = '/content/data'                     # local disk: much faster than reading from Drive
os.environ['CKPT_DIR'] = f'{DRIVE_DIR}/checkpoints'

## 2. Get the code

In [ ]:
token = ''
try:
    from google.colab import userdata
    token = userdata.get('GH_TOKEN') or ''
except Exception:
    pass
url = f'https://{token}@github.com/{REPO}.git' if token else f'https://github.com/{REPO}.git'
if not os.path.isdir('/content/iad'):
    !git clone -q -b {BRANCH} {url} /content/iad
else:
    !git -C /content/iad pull -q
%cd /content/iad
!git log --oneline -1

In [ ]:
!pip install -q -r requirements.txt
!apt-get -qq install -y rsync > /dev/null

## 3. Data: restore the bundle from Drive, or build it once

In [ ]:
own_bundle = f'{DRIVE_DIR}/iad_data.tar'
shared_bundle = '/content/drive/MyDrive/iad_data.tar'          # shortcut to the team's view-only copy
bundle = next((b for b in (own_bundle, shared_bundle) if os.path.exists(b)), None)
os.makedirs(os.environ['DATA_ROOT'], exist_ok=True)
if bundle:
    print('restoring data from', bundle)
    !tar -xf "{bundle}" -C $DATA_ROOT                          # read-only: the bundle is never modified
!bash scripts/setup.sh      # fresh: downloads + prepares everything; restored: only code + checkpoints
if not bundle:
    !bash scripts/pack_data.sh "{own_bundle}"                  # saved to your own Drive folder only
!ls $DATA_ROOT $DATA_ROOT/nyu

## 4. Smoke test (~1 min)

In [ ]:
%cd /content/iad/tools
!python infer.py $DATA_ROOT/nyu/rgb/000500.png --outdir /content/iad/results/infer
!python eval_nyu.py --split val --limit 10 --tag colab_smoke

## 5. Baseline evaluations (optional, ~5 min)
Use `--split val` while tuning; `--split test` only for final numbers.

In [ ]:
!python eval_nyu.py --split val  --gt filled
!python eval_nyu.py --split test --gt filled
!python eval_nyu.py --split test --gt raw
!python eval_da2k.py

## 6. Save results to Drive

In [ ]:
import shutil, time
dst = f'{DRIVE_DIR}/results/{time.strftime("%Y%m%d-%H%M%S")}'
shutil.copytree('/content/iad/results', dst, ignore=shutil.ignore_patterns('*.ply', '*.npy'))
print('saved to', dst)